In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType
from pyspark.sql.functions import round

In [2]:
spark = SparkSession.builder.master("local[*]").appName("Test").getOrCreate()

custom_schema = StructType([
    StructField("Date", DateType(), False),
    StructField("Steps", IntegerType(), False),
    StructField("Distance", DoubleType(), False),
    StructField("Floors Ascended", IntegerType(), False)
])


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/06 19:42:46 WARN Utils: Your hostname, kachi, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/06 19:42:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 19:42:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("./data.csv", header=True, inferSchema=True)
df.show(5)

+----------+-----+--------+---------------+
|      Date|Steps|Distance|Floors Ascended|
+----------+-----+--------+---------------+
|06/04/2026|  146|     0.1|              0|
|05/04/2026|  620|     0.2|              0|
|04/04/2026| 4406|     1.8|              7|
|03/04/2026| 2415|     1.0|              4|
|02/04/2026|  773|     0.3|              0|
+----------+-----+--------+---------------+
only showing top 5 rows


In [4]:
df.schema

StructType([StructField('Date', StringType(), True), StructField('Steps', IntegerType(), True), StructField('Distance', DoubleType(), True), StructField('Floors Ascended', IntegerType(), True)])

In [5]:
df.select("Date", df.Distance, round(df.Distance * 1.60934, 2).alias("Distance in KM")).show(5)

+----------+--------+--------------+
|      Date|Distance|Distance in KM|
+----------+--------+--------------+
|06/04/2026|     0.1|          0.16|
|05/04/2026|     0.2|          0.32|
|04/04/2026|     1.8|           2.9|
|03/04/2026|     1.0|          1.61|
|02/04/2026|     0.3|          0.48|
+----------+--------+--------------+
only showing top 5 rows


In [6]:
df = df.withColumnRenamed("Distance", "Distance in Miles")
df.show(5)

+----------+-----+-----------------+---------------+
|      Date|Steps|Distance in Miles|Floors Ascended|
+----------+-----+-----------------+---------------+
|06/04/2026|  146|              0.1|              0|
|05/04/2026|  620|              0.2|              0|
|04/04/2026| 4406|              1.8|              7|
|03/04/2026| 2415|              1.0|              4|
|02/04/2026|  773|              0.3|              0|
+----------+-----+-----------------+---------------+
only showing top 5 rows


In [7]:
df.filter(df.Steps < 50).show(5)

+----------+-----+-----------------+---------------+
|      Date|Steps|Distance in Miles|Floors Ascended|
+----------+-----+-----------------+---------------+
|29/03/2026|   49|              0.0|              0|
|02/01/2026|    0|              0.0|              0|
|25/10/2025|    9|              0.0|              0|
|20/07/2024|   32|              0.0|              0|
|10/07/2024|   24|              0.0|              0|
+----------+-----+-----------------+---------------+
only showing top 5 rows


In [8]:
import pyspark.sql.functions as F

In [9]:
len(dir(F))

554

In [10]:
df.select(
    "Date",
    "Steps",
    F.when(df.Steps > 100, "Active").otherwise("inactive").alias("Activity Level"),
).show(5)

+----------+-----+--------------+
|      Date|Steps|Activity Level|
+----------+-----+--------------+
|06/04/2026|  146|        Active|
|05/04/2026|  620|        Active|
|04/04/2026| 4406|        Active|
|03/04/2026| 2415|        Active|
|02/04/2026|  773|        Active|
+----------+-----+--------------+
only showing top 5 rows


In [ ]:
spark.stop()